# Sideband Analysis for the ANNIE NCQE cross section

This script samples the >2 sigma inter-bunch space sideband to constrain the external + skyshine rate. The script uses MC samples + full FY22-23 data (2.57e20 POT) used also for the cross section extraction.

This script requires two passes:
1. Analyze the entire spill to extract the estimated skyshine events
2. Applying the 1/3 time cut with the estimated skyshine counts to produce sideband plots

In [1]:
from lib.imports import *
from lib.constants import *
from lib.reco_functions import (
    reco_energy,
    pairwise_relative_direction,
    is_inside_rotated_parabola,
)

# Because NCQE signal is poor, it doesn't really matter which GENIE model we use
# The extracted skyshine fit is near-identical; we can use the Empirical model for consistency

# REQUIRES USER INPUT (specify path to MC files)
MC_directory = '../MC_GENIE-O16/'   # or 'MC_GENIE-Base/'
plots_dir = 'plots/sidebands/'      # directory to hold plots
os.system('mkdir -p ' + plots_dir)  # auto-generate plots directory if not present

POT = MC_POT_Empirical   # MC POT, 4.41733e20

print('\nWill use ' + MC_directory + ' for the MC samples, and ' + str(plots_dir) +' for the plots output directory')

data_directory = '../../FY22_23/'
csv_file = 'lib/POT_trigger_FY22_23_summary.csv'   # contains trigger counts and POT for on-beam data
offbeam_directory = '../../off_beam/'
csv_file_ext = 'lib/offbeam_summary.csv'           # contains trigger counts for off-beam data

# simulated skyshine files
skyshine_file_names = [
    '../../skyshine_100k_no_ext_clusters.root',
    '../../skyshine_100k_no_ext_clusters_2.root',
    '../../skyshine_100k_no_ext_clusters_3.root'
]

# Because the skyshine is an inversion of the bunch cut selection, we re-define the bunch cutting function
def bunch_cutting(cluster_time, centers):
    avg_diff = bunch_offset_mapping
    return np.any(np.abs( (centers - avg_diff) - cluster_time) < (2*bunch_sigma))


Will use ../MC_GENIE-O16/ for the MC samples, and plots/sidebands/ for the plots output directory


In [ ]:
# on-beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_BEAM_DATA/)

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

file_names = [file for file in os.listdir(data_directory) if os.path.isfile(os.path.join(data_directory, file))]
print('There are: ', len(file_names), ' files\n')

# Extract all run numbers from the current files
run_numbers = [extract_run_number(f) for f in file_names]

# POT file
full_pot_df = pd.read_csv(csv_file)

# Filter the dataframe to only include runs that are in the <STRING>/ directory
pot_df = full_pot_df[full_pot_df['RUN'].isin(run_numbers)]

# extract total POT and triggers for only the matched runs
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
only_prompt_cluster = []
MRD_activity = []
Veto_activity = []
Vtx_reco = []
Vty_reco = []
Vtz_reco = []
hitX = []
hitY = []
hitZ = []
hitT = []
hitPE = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(data_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity.append(Event["MRD_activity"].array(library="np"))
        Veto_activity.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco.append(Event["recoVtx"].array(library="np"))
        Vty_reco.append(Event["recoVty"].array(library="np"))
        Vtz_reco.append(Event["recoVtz"].array(library="np"))
        
        hitX.append(Event["hitX"].array())
        hitY.append(Event["hitY"].array())
        hitZ.append(Event["hitZ"].array())
        hitT.append(Event["hitT"].array())
        hitPE.append(Event["hitPE"].array())

        #print(len(cluster_time[-1]), "clusters")
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
only_prompt_cluster = np.concatenate(only_prompt_cluster)
MRD_activity = np.concatenate(MRD_activity)
Veto_activity = np.concatenate(Veto_activity)
Vtx_reco = np.concatenate(Vtx_reco)
Vty_reco = np.concatenate(Vty_reco)
Vtz_reco = np.concatenate(Vtz_reco)

# Jagged arrays
hitX = ak.concatenate(hitX)
hitY = ak.concatenate(hitY)
hitZ = ak.concatenate(hitZ)
hitT = ak.concatenate(hitT)
hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

In [ ]:
# off beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_OFFBEAM_DATA/)
file_names_ext = [file for file in os.listdir(offbeam_directory) if os.path.isfile(os.path.join(offbeam_directory, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT file
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
only_prompt_cluster_ext = []
MRD_activity_ext = []
Veto_activity_ext = []
Vtx_reco_ext = []
Vty_reco_ext = []
Vtz_reco_ext = []
hitX_ext = []
hitY_ext = []
hitZ_ext = []
hitT_ext = []
hitPE_ext = []
runs_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names_ext), ')')
    
    with uproot.open(os.path.join(offbeam_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster_ext.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
        Veto_activity_ext.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco_ext.append(Event["recoVtx"].array(library="np"))
        Vty_reco_ext.append(Event["recoVty"].array(library="np"))
        Vtz_reco_ext.append(Event["recoVtz"].array(library="np"))
        
        hitX_ext.append(Event["hitX"].array())
        hitY_ext.append(Event["hitY"].array())
        hitZ_ext.append(Event["hitZ"].array())
        hitT_ext.append(Event["hitT"].array())
        hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
only_prompt_cluster_ext = np.concatenate(only_prompt_cluster_ext)
MRD_activity_ext = np.concatenate(MRD_activity_ext)
Veto_activity_ext = np.concatenate(Veto_activity_ext)
Vtx_reco_ext = np.concatenate(Vtx_reco_ext)
Vty_reco_ext = np.concatenate(Vty_reco_ext)
Vtz_reco_ext = np.concatenate(Vtz_reco_ext)

# Jagged arrays
hitX_ext = ak.concatenate(hitX_ext)
hitY_ext = ak.concatenate(hitY_ext)
hitZ_ext = ak.concatenate(hitZ_ext)
hitT_ext = ak.concatenate(hitT_ext)
hitPE_ext = ak.concatenate(hitPE_ext)

print(total_ext_triggers, 'triggers')

In [ ]:
# Load MC samples

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_CV_weight         = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])
    
    MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)
MC_CV_weight       = np.concatenate(MC_CV_weight)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

In [ ]:
# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

for file_n in skyshine_file_names:

    root = uproot.open(file_n)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
    skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
    skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
    skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
    skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
    skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))

    skyshine_hitX.append(T['hitX'].array())
    skyshine_hitY.append(T['hitY'].array())
    skyshine_hitZ.append(T['hitZ'].array())
    skyshine_hitT.append(T['hitT'].array())
    skyshine_hitPE.append(T['hitPE'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']

    # map trigger-level quantities onto clusters via CEN
    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    skyshine_MRD_hitT.append(mrd_t[CEN])
    skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')

In [ ]:
# First, MC

# the dimensions of the saved arrays are as follows:
# [0]: nuNCQE (signal)
# [1]: NCother
# [2]: CC
# [3]: nubarNCQE
# [4]: Out-of-FV (OOFV)
# [5]: External (interactions outside of the water tank entirely)

MC_reco_E = [[], [], [], [], [], []]
MC_CV_x = [[], [], [], [], [], []]
MC_CV_y = [[], [], [], [], [], []]
MC_CV_z = [[], [], [], [], [], []]
MC_bunch_times = [[], [], [], [], [], []]
MC_vertex_x = [[], [], [], [], [], []]
MC_vertex_y = [[], [], [], [], [], []]
MC_vertex_z = [[], [], [], [], [], []]
MC_hits = [[], [], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], [], []]


FIRST_PASS = True      # see intro cell: first passthrough we disable the 1/3 timing cut to extract the skyshine contribution

if FIRST_PASS == True:
    print('NO ONE THIRD TIMING CUT APPLIED')
else:
    print('APPLYING ONE THIRD TIMING CUT')

for i in trange(len(MC_cluster_time), desc = 'applying NCQE selection cuts to MC...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if MC_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if MC_bunchTimes[i] > time_to_prompt_end:
        continue
    # minimum 10 hits
    if MC_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    # No MRD/FMV correlated activity
    if (
        MC_TankMRDCoinc[i] == 1
        or MC_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in MC_MRD_hitT[i])
        or any(0 <= t <= 300 for t in MC_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill
    if MC_bunchTimes[i] > bunch_time_end:
        continue
    if MC_bunchTimes[i] < bunch_time_start:
        continue

    if FIRST_PASS == False:
        # Add earlier time cut (reject anything later than the 25th bunch center)
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE FIRST_PASS
        if MC_bunchTimes[i] > MC_third_spill_cut:
            continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(MC_cluster_time)):        # look at the subsequent clusters in this event
        if MC_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if MC_bunchTimes[k] < time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if MC_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut (FV defined in first cell)
    if MC_recoVtx[i] < -50:
        continue
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(MC_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(MC_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    # INVERT BUNCH CUT FOR SIDEBAND
    if bunch_cutting(MC_bunchTimes[i], centers) == True:    # if the time is inside the 2-sigma cut, reject it
        continue
        
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    # event categorization
    if MC_External[i] == True:
        indy = 5
        MC_reco_E[indy].append(recoE)
        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
        MC_bunch_times[indy].append(MC_bunchTimes[i])
        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
        MC_hits[indy].append(MC_cluster_Hits[i])
        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
    else:
        if MC_FV[i] == True:
            if MC_NC[i] == 1:       # NC
                if MC_QEL[i] == 1:
                    if MC_Z[i] == 8:
                        if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                            # nuNCQE
                            indy = 0
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                        elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                            # nubarNCQE
                            indy = 3
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                    else:
                        # NC elastic on H (NCother)
                        indy = 1
                        MC_reco_E[indy].append(recoE)
                        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                        MC_bunch_times[indy].append(MC_bunchTimes[i])
                        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                        MC_hits[indy].append(MC_cluster_Hits[i])
                        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                else:
                    # NC, not QEL (NCother)
                    indy = 1
                    MC_reco_E[indy].append(recoE)
                    MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                    MC_bunch_times[indy].append(MC_bunchTimes[i])
                    MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                    MC_hits[indy].append(MC_cluster_Hits[i])
                    MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
            else:
                # CC
                indy = 2
                MC_reco_E[indy].append(recoE)
                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
        else:
            # out-of-FV
            indy = 4
            MC_reco_E[indy].append(recoE)
            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
            MC_bunch_times[indy].append(MC_bunchTimes[i])
            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
            MC_hits[indy].append(MC_cluster_Hits[i])
            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
          

print('done')

In [ ]:
# Apply selection cuts for On-Beam data

reco_E = []
CV_x = []
CV_y = []
CV_z = []
bunch_times = []
vertex_x = []
vertex_y = []
vertex_z = []
hits = []
reco_runs = []


FIRST_PASS = True

if FIRST_PASS == True:
    print('NO ONE THIRD TIMING CUT APPLIED')
else:
    print('APPLYING ONE THIRD TIMING CUT')


for i in trange(len(cluster_time), desc = 'applying NCQE selection cuts to on-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0:
        continue

    # bunch cutting
    if cluster_time[i] < bunch_time_start_data:
        continue
    if cluster_time[i] > bunch_time_end_data:
        continue
            
    if FIRST_PASS == False:
        if cluster_time[i] > data_third_spill_cut:
            continue
        
    # INVERT BUNCH CUT FOR SIDEBAND
    if bunch_cutting(cluster_time[i], centers_data) == True:
        continue
    
    # CCB cut
    if cluster_Qb[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco[i] < -50:
        continue
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # event selection
    reco_E.append(recoE)
    CV_x.append(cw[0]); CV_y.append(cw[1]); CV_z.append(cw[2])
    bunch_times.append(cluster_time[i])
    vertex_x.append(Vtx_reco[i]); vertex_y.append(Vty_reco[i]); vertex_z.append(Vtz_reco[i])
    hits.append(cluster_Hits[i])
    reco_runs.append(runs[i])
          

print('done')

In [ ]:
# ext data

ext_reco_E = []
ext_CV_x = []
ext_CV_y = []
ext_CV_z = []
ext_bunch_times = []
ext_vertex_x = []
ext_vertex_y = []
ext_vertex_z = []
ext_hits = []
ext_reco_runs = []

ext_outside_bunch_times = []


FIRST_PASS = True

if FIRST_PASS == True:
    print('NO ONE THIRD TIMING CUT APPLIED')
else:
    print('APPLYING ONE THIRD TIMING CUT')


for i in trange(len(cluster_time_ext), desc = 'applying NCQE selection cuts to off-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster_ext[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0:
        continue
        
    # CCB cut
    if cluster_Qb_ext[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco_ext[i] < -50:
        continue
    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco_ext[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge_ext[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # bunch cutting
    if cluster_time_ext[i] < bunch_time_start_data:
        continue
    if cluster_time_ext[i] > bunch_time_end_data:
        continue

    if FIRST_PASS == False:
        if cluster_time_ext[i] > data_third_spill_cut:
            continue

    if bunch_cutting(cluster_time_ext[i], centers_data) == True:
        ext_outside_bunch_times.append(cluster_time_ext[i])
        continue
        
    # event selection
    ext_reco_E.append(recoE)
    ext_CV_x.append(cw[0]); ext_CV_y.append(cw[1]); ext_CV_z.append(cw[2])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_vertex_x.append(Vtx_reco_ext[i]); ext_vertex_y.append(Vty_reco_ext[i]); ext_vertex_z.append(Vtz_reco_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

In [ ]:
skyshine_reco_E = []
skyshine_CV_x = []
skyshine_CV_y = []
skyshine_CV_z = []
skyshine_bunch_times = []
skyshine_vertex_x = []
skyshine_vertex_y = []
skyshine_vertex_z = []
skyshine_hits = []

FIRST_PASS = True

if FIRST_PASS == True:
    print('NO ONE THIRD TIMING CUT APPLIED')
else:
    print('APPLYING ONE THIRD TIMING CUT')


# do not apply timing cuts
for i in trange(len(skyshine_cluster_time), desc = 'applying NCQE selection cuts to simulated skyshine...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if skyshine_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window (just an approximation)
    if skyshine_cluster_time[i] > time_to_prompt_end:
        continue
    # minimum 10 hits
    if skyshine_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue

    if skyshine_cluster_time[i] > bunch_time_end:
        continue
        
    # Cluster time within 1.6 us beam spill of particle creation (t0 = 0 for these simulations)
    if FIRST_PASS == False:
        if skyshine_cluster_time[i] > MC_third_spill_cut:
            continue
        
    # No bunch cutting applied
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(skyshine_cluster_time)):        # look at the subsequent clusters in this event
        if skyshine_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if skyshine_cluster_time[k] < time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if skyshine_recoVtx[i] < -50:
        continue
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    skyshine_reco_E.append(recoE)
    skyshine_CV_x.append(cw[0]); skyshine_CV_y.append(cw[1]); skyshine_CV_z.append(cw[2])
    skyshine_bunch_times.append(skyshine_cluster_time[i])
    skyshine_vertex_x.append(skyshine_recoVtx[i]); 
    skyshine_vertex_y.append(skyshine_recoVty[i]);
    skyshine_vertex_z.append(skyshine_recoVtz[i])
    skyshine_hits.append(skyshine_cluster_Hits[i])
    

print('done')

In [ ]:
# 1. Scaling factors
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
POT_MC = POT/(1e20)
POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers

# 2. Chain all bunch times to a common t0
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def get_normalized_times(times, runs, weights=None, mc=False):

    norm_times = []
    norm_weights = []

    for i, (t, r) in enumerate(zip(times, runs)):

        if mc:
            tnorm = t - bunch_time_start
        else:
            tnorm = t - bunch_time_start_data

        norm_times.append(tnorm)

        if weights is not None:
            norm_weights.append(weights[i])

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)

    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(6):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)


# 3. Calculation of radial distance of the reco vertex
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
# For MC (6 categories)
radius_mc = [np.sqrt(np.array(MC_vertex_x[i])**2 + np.array(MC_vertex_z[i])**2) for i in range(6)]
# For On-Beam Data
radius_data = np.sqrt(np.array(vertex_x)**2 + np.array(vertex_z)**2)
# For Off-Beam (EXT) Data
radius_ext = np.sqrt(np.array(ext_vertex_x)**2 + np.array(ext_vertex_z)**2)


mc_stack = [
    bunches_80_mc[5],   # External MC (dirt)  <- scaled by ALPHA_SCALE
    bunches_80_mc[4],   # Out-of-FV
    bunches_80_mc[2],   # CC
    bunches_80_mc[1],   # NCother
    bunches_80_mc[3],   # nubarNCQE
    bunches_80_mc[0],   # nuNCQE    (top)
]
mc_weights_stack = [
    bunches_80_weights[5],
    bunches_80_weights[4],
    bunches_80_weights[2],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[0],
]


radius_skyshine = np.sqrt(np.array(skyshine_vertex_x)**2 + np.array(skyshine_vertex_z)**2)
    
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *

print('done')

In [ ]:
# Run for skyshine estimation (fit to the rise in activity)
# Skip this if it's not your first pass through


def delayed_linear(t, C, m, t0):
    # C: Flat baseline events/bin
    # m: Skyshine slope
    # t0: Turn-on time in ns
    return np.where(t < t0, C, C + m * (t - t0))

def plain_linear(t, C, m):
    # C: intercept (events/bin), m: slope (evts/bin)/ns -- no turn-on point
    return C + m * t


def calculate_delayed_skyshine_rate(bins, prediction_data, weights, on_beam_data,
                                     model='piecewise', t_skip=0.0):
    """
    model  : 'piecewise' (default) -- flat baseline C, then C + m*(t-t0) after
             a fitted turn-on time t0 (3 free params: C, m, t0).
             'linear' -- plain C + m*t across the whole range, no turn-on
             point (2 free params: C, m).
    t_skip : exclude any event with t < t_skip from both data and prediction,
             before histogramming/fitting (e.g. to drop the low-POT first
             "baby bunch" -- see the driver cell for how T_SKIP_FIRST_BUNCH
             is computed).
    """
    if model not in ('piecewise', 'linear'):
        raise ValueError(f"model must be 'piecewise' or 'linear', got {model!r}")

    if t_skip > 0:
        on_beam_data = np.asarray(on_beam_data)
        on_beam_data = on_beam_data[on_beam_data >= t_skip]
        filtered_pred, filtered_weights = [], []
        for arr, w in zip(prediction_data, weights):
            arr = np.asarray(arr)
            w = np.asarray(w)
            mask = arr >= t_skip
            filtered_pred.append(arr[mask])
            filtered_weights.append(w[mask])
        prediction_data = filtered_pred
        weights = filtered_weights

    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_width = bins[1] - bins[0]

    # 1. Build Histograms
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    data_err[data_err == 0] = 1.0

    pred_counts = np.zeros_like(data_counts, dtype=float)
    pred_err_sq = np.zeros_like(data_counts, dtype=float)
    for i in range(len(prediction_data)):
        c, _ = np.histogram(prediction_data[i], bins=bins, weights=weights[i])
        w2, _ = np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)
        pred_counts += c
        pred_err_sq += w2
    pred_err = np.sqrt(pred_err_sq)
    pred_err[pred_err == 0] = 1.0

    # 2. Pick the model
    if model == 'piecewise':
        fit_func    = delayed_linear
        p0_data     = [100.0, 0.2, 400.0]
        bounds_data = ([0, 0, 0], [np.inf, np.inf, bins[-1]])
        p0_pred     = [100.0, 0.0, 400.0]
        bounds_pred = ([0, -np.inf, 0], [np.inf, np.inf, bins[-1]])
    else:  # 'linear'
        fit_func    = plain_linear
        p0_data     = [100.0, 0.2]
        bounds_data = ([0, -np.inf], [np.inf, np.inf])
        p0_pred     = [100.0, 0.0]
        bounds_pred = ([0, -np.inf], [np.inf, np.inf])

    # 3. Perform the Fits
    popt_data, pcov_data = curve_fit(
        fit_func, bin_centers, data_counts,
        sigma=data_err, absolute_sigma=True,
        p0=p0_data, bounds=bounds_data,
    )
    popt_pred, pcov_pred = curve_fit(
        fit_func, bin_centers, pred_counts,
        sigma=pred_err, absolute_sigma=True,
        p0=p0_pred, bounds=bounds_pred,
    )

    err_m_data = np.sqrt(pcov_data[1, 1])
    err_m_pred = np.sqrt(pcov_pred[1, 1])

    if model == 'piecewise':
        C_data, m_data, t0_data = popt_data
        C_pred, m_pred, t0_pred = popt_pred
        err_t0_data = np.sqrt(pcov_data[2, 2])
    else:
        C_data, m_data = popt_data
        C_pred, m_pred = popt_pred
        t0_data = None
        err_t0_data = None

    # 4. Calculate True Rate
    delta_m = m_data - m_pred
    err_delta_m = np.sqrt(err_m_data**2 + err_m_pred**2)
    rate_increase = delta_m / bin_width
    err_rate_increase = err_delta_m / bin_width

    print("\n" + "="*50)
    print(f" DELAYED SKYSHINE EXTRACTION ({model.upper()})")
    print("="*50)
    if model == 'piecewise':
        print(f"Data Skyshine Turn-on Time: {t0_data:.1f} \u00b1 {err_t0_data:.1f} ns")
    print(f"Data Baseline (C):          {C_data:.1f} events/bin")
    label = "Data Slope after t0:       " if model == 'piecewise' else "Data Slope:                "
    print(f"{label} {m_data:.5f} \u00b1 {err_m_data:.5f} (evts/bin)/ns")
    print("-" * 50)
    print(f"Excess Slope:               {delta_m:.5f} \u00b1 {err_delta_m:.5f} (evts/bin)/ns")
    print(f"True Skyshine Acceleration: {rate_increase:.6f} \u00b1 {err_rate_increase:.6f} events/ns\u00b2")
    if t_skip > 0:
        print(f"(fit excluded all events with t < {t_skip:.1f} ns)")
    print("="*50 + "\n")

    return popt_data, t0_data, err_t0_data, rate_increase, err_rate_increase


def plot_sideband_comparison(bins, mc_data, mc_weights, on_beam_data, off_beam_data,
                             pot_mc, pot_data, trig_ext, trig_data,
                             axis_label, y_axis_label, x_limit, y_limit, xaxis_minor,
                             plot_name, legend_columns=3,
                             model='piecewise', t_skip=0.0, alpha_scale=0.18):

    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext

    # 2. Prepare Prediction Stack (MC + Off-Beam)
    # Order: [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE, Off-Beam Data]
    prediction_data = mc_data + [off_beam_data]

    # Per-event GENIE weights * POT scale
    weights = [np.array(mc_weights[i]) * f_pot for i in range(len(mc_data))]
    weights.append(np.ones(len(off_beam_data)) * f_ext)

    # --- DEBUG PRINTS ---
    print(f"f_pot scaling factor: {f_pot:.3f}")
    print(f"Raw Dirt events before weights: {len(prediction_data[0])}")
    print(f"Dirt Yield BEFORE scale: {np.sum(weights[0]):.1f}")

    # 3. Aesthetics
    labels_base = ['External (MC)', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 6 + ['black']

    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * alpha_scale
            print(f"Applied {alpha_scale} scale to index {i} ({label})")

    print(f"Dirt Yield AFTER scale: {np.sum(weights[0]):.1f}")
    print(f"Off-Beam Yield: {np.sum(weights[-1]):.1f}")

    # --- omit the first bunch (or any t < t_skip) from BOTH the plot and the
    # fit, so what's shown and what's fit stay consistent ---
    if t_skip > 0:
        on_beam_data = np.asarray(on_beam_data)
        on_beam_data = on_beam_data[on_beam_data >= t_skip]
        filtered_pred, filtered_weights = [], []
        for arr, w in zip(prediction_data, weights):
            arr = np.asarray(arr)
            w = np.asarray(w)
            mask = arr >= t_skip
            filtered_pred.append(arr[mask])
            filtered_weights.append(w[mask])
        prediction_data = filtered_pred
        weights = filtered_weights
        print(f"Excluding all events with t < {t_skip:.1f} ns (first bunch) from plot + fit")

    # skyshine rate calculation (already-filtered data -> t_skip=0.0 here)
    popt_data, t0_, t0_err_, accel_, accel_err_  = calculate_delayed_skyshine_rate(bins, prediction_data, weights, on_beam_data, model=model)

    # --- Calculate Chi-Squared for the Fit ---
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    if model == 'piecewise':
        C_data, m_data, t0_data = popt_data
        fit_at_centers = np.where(bin_centers < t0_data, C_data, C_data + m_data * (bin_centers - t0_data))
        n_params = 3
    else:
        C_data, m_data = popt_data
        t0_data = None
        fit_at_centers = C_data + m_data * bin_centers
        n_params = 2

    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    valid_bins = data_counts > 0

    chi2 = np.sum(((data_counts[valid_bins] - fit_at_centers[valid_bins]) / data_err[valid_bins])**2)
    ndof = np.sum(valid_bins) - n_params
    print(chi2, ndof, chi2/ndof)
    # ----------------------------------------------

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]

    # 4. Math for Residuals and Errors
    bin_widths = np.diff(bins)

    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)

    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill (No edges to avoid the bar-graph look)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            label=labels, color=colors, linewidth=0)

    # B. The Outline
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2,
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)

    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]

    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)

    ### Overlay the fit (piecewise or linear)
    x_fit = np.linspace(x_limit[0], x_limit[1], 500)
    if model == 'piecewise':
        y_fit = np.where(x_fit < t0_data, C_data, C_data + m_data * (x_fit - t0_data))
    else:
        y_fit = C_data + m_data * x_fit

    ax.plot(x_fit, y_fit, color='magenta', linestyle='--', linewidth=2.5)
    ax.text(0.15, 0.7, r"$\chi^2 / \text{dof} = $" + f"{chi2:.1f}/{ndof}", transform=ax.transAxes)
    ### END

    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))

    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_ylim([-1.2, 4])
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))

    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)

    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)

    plt.tight_layout()
    plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()

    return t0_, t0_err_, accel_, accel_err_


# --- driver ---
MODEL = 'piecewise'   # or 'linear'

n_bunches_per_bin = 5
bin_width = 18.936 * n_bunches_per_bin

# Skip the first ("baby") bunch: exclude everything before bunch 1's exclusion
# zone starts, in the same bunches_80_data-normalized frame (t=0 at spill
# start). This removes both the pre-bunch-0 buffer window and the
# bunch0<->bunch1 gap -- i.e. every sideband window adjacent to bunch 0.
# Set to 0.0 to include everything, as before.
T_SKIP_FIRST_BUNCH = (centers_data[1] - bunch_offset_mapping - bunch_sigma) - bunch_time_start_data

# Start the first bin where real sideband data actually resumes after
# excluding bunch 0 -- the END of bunch 1's own exclusion zone, not
# T_SKIP_FIRST_BUNCH itself (which is where that zone *starts*). This keeps
# the first bin's live time consistent with every other bin's.
bin_start = T_SKIP_FIRST_BUNCH + bunch_sigma

# Use the true spill end rather than a hand-picked bunch count, and floor()
# the bin count so we never overshoot into a trailing phantom-empty bin
# (same failure mode fixed earlier for the full-spill case).
bin_end = bunch_time_end_data - bunch_time_start_data

n_bins = int(np.floor((bin_end - bin_start) / bin_width))
bins = bin_start + np.arange(n_bins + 1) * bin_width

print(f"{len(bins)-1} bins of width {bin_width:.2f} ns, spanning [{bins[0]:.1f}, {bins[-1]:.1f}] ns "
      f"(true spill end: {bin_end:.1f} ns)")

# Reorder MC for the stack
mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2],
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3
    }

t0_, t0_err_, accel_, accel_err_ = plot_sideband_comparison(
    bins=bins,
    mc_data=mc_stack,
    mc_weights=mc_weights_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / ' + str(n_bunches_per_bin) + ' BNB bunches',
    x_limit=[bin_start, bin_end - bin_width],
    y_limit=[0, 200],
    xaxis_minor=50,
    plot_name=plots_dir + 'skyshine_timing_fit.png',
    model=MODEL,
    t_skip=T_SKIP_FIRST_BUNCH,
    alpha_scale=ALPHA_SCALING,
    **common_args
)

print('done')

In [ ]:
# turn the fit results into an actual estimate

# Skip if not your first time through

# N_skyshine = 1/2*A(T_cut - t0)^2

#t_cut = (centers_data[25] - 0.785 + 3.5) - bunch_time_start_data
t_cut = data_third_spill_cut - bunch_time_start_data
print(t_cut)

def calculate_skyshine_in_window(t_cut, t0, err_t0, A, err_A):
    if t_cut <= t0:
        return 0.0, 0.0
    
    delta_t = t_cut - t0
    
    # Calculate expected events
    N_events = 0.5 * A * (delta_t**2)
    
    # Error propagation
    # Partial derivative wrt A: 0.5 * delta_t^2
    term_A = (0.5 * (delta_t**2) * err_A)**2
    # Partial derivative wrt t0: -A * delta_t
    term_t0 = (A * delta_t * err_t0)**2
    
    err_N = np.sqrt(term_A + term_t0)
    
    print(f"Time Cut: {t_cut} ns")
    print(f"Expected Skyshine Events: {N_events:.1f} ± {err_N:.1f}")
    print(f"Expected Skyshine Events (signal region): {N_events*EXPOSURE_SCALE:.1f} ± {err_N*EXPOSURE_SCALE:.1f}")

    return N_events

# Example usage with your 5-bunch results
SKYSHINE_FITTING_RESULTS = calculate_skyshine_in_window(
    t_cut=t_cut, 
    t0=t0_, err_t0=t0_err_, 
    A=accel_, err_A=accel_err_
)

In [ ]:
# At this point (if it is your first pass through), you should re-run the cells with FIRST_PASS == False

In [ ]:
# We can first run a 1D dirt normalization check to assess our dirt uncertainty
# This is defined by fixing the contribution of skyshine (found earlier)
# and, similarily to the CC sideband, scaling the dirt MC events so that the total event rate matches the data

# We don't actively apply this scaling for plotting the observables,
# but instead can derive a dirt scaling factor for this sideband to compare to the CC scaling factor
# The uncertainty due to dirt is then determined by re-extracting the XS with the skyshine sideband dirt scaling factor

# ---------------------------------------------------------
# 1D RATE NORMALIZATION (Total Yield Matching)
# ---------------------------------------------------------
def compute_dirt_scale_with_skyshine(
        mc_stack,
        mc_weights,
        data_events,
        ext_events,
        skyshine_events,
        expected_skyshine_yield,
        pot_mc,
        pot_data,
        trig_ext,
        trig_data,
        verbose=True):

    # ---------------------------------------------------------
    # Scaling factors
    # ---------------------------------------------------------
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext

    # ---------------------------------------------------------
    # Totals
    # ---------------------------------------------------------
    total_data = len(data_events)

    # Fixed MC (everything except dirt = index 0) -- per-event GENIE weights,
    # not raw counts (was sum(len(mc_stack[i]) * f_pot ...))
    total_fixed_mc = sum(np.sum(mc_weights[i]) * f_pot
                         for i in range(1, len(mc_stack)))

    # Off-beam -- real data, no per-event weight to apply, just the
    # trigger-ratio scale
    total_off_beam = len(ext_events) * f_ext

    # Skyshine (forced to expected yield, just like your plot code)
    if skyshine_events is not None and len(skyshine_events) > 0:
        total_skyshine = expected_skyshine_yield
    else:
        total_skyshine = 0.0

    total_fixed = total_fixed_mc + total_off_beam + total_skyshine

    # Unscaled dirt MC -- per-event weights, not raw count
    total_ext_mc = np.sum(mc_weights[0]) * f_pot

    # Solve for dirt scale
    if total_ext_mc <= 0:
        rate_scale_factor = 0.0
    else:
        rate_scale_factor = (total_data - total_fixed) / total_ext_mc

    # ---------------------------------------------------------
    # Printout
    # ---------------------------------------------------------
    if verbose:
        print("\n" + "="*45)
        print(" 1D RATE NORMALIZATION (WITH SKYSHINE)")
        print("="*45)
        print(f"Total Data Events:         {total_data}")
        print(f"Fixed MC (no dirt):        {total_fixed_mc:.1f}")
        print(f"Off-beam contribution:     {total_off_beam:.1f}")
        print(f"Skyshine (fixed):          {total_skyshine:.1f}")
        print(f"Total Fixed Backgrounds:   {total_fixed:.1f}")
        print(f"Total Unscaled Ext MC:     {total_ext_mc:.1f}")
        print("-" * 45)
        print(f"Calculated Dirt Scale:     {rate_scale_factor:.3f}")
        print("="*45 + "\n")

    return rate_scale_factor


rate_scale_factor = compute_dirt_scale_with_skyshine(
    mc_stack=mc_stack,
    mc_weights=mc_weights_stack,
    data_events=bunches_80_data,
    ext_events=bunches_80_ext,
    skyshine_events=skyshine_bunch_times,
    expected_skyshine_yield=SKYSHINE_FITTING_RESULTS,
    pot_mc=POT_MC,
    pot_data=POT_DATA,
    trig_ext=TRIG_EXT,
    trig_data=TRIG_DATA
)

In [ ]:
# Plotting machinery

def plot_with_skyshine(bins, mc_data, on_beam_data, off_beam_data, skyshine_data, 
                       pot_mc, pot_data, trig_ext, trig_data, expected_skyshine_yield,
                       axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                       plot_name, legend_columns=3,
                       mc_weights_stack=None, 
                       DEBUG=False, ALPHA_SCALE=ALPHA_SCALING, SAVE_PLOT=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    ### NEW: Calculate the weight required to force the skyshine MC to exactly match your expected yield
    if len(skyshine_data) > 0:
        f_sky = expected_skyshine_yield / len(skyshine_data)
    else:
        f_sky = 0.0
    
    # 2. Prepare Prediction Stack (MC + Skyshine + Off-Beam)
    ### NEW: We inject Skyshine right after External (MC) [which is index 0]
    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    
    ### NEW: We build the weights array to match the new 8-element prediction_data order
    def mc_w(i):
        if mc_weights_stack is not None:
            return np.array(mc_weights_stack[i]) * f_pot
        return np.ones(len(mc_data[i])) * f_pot

    weights = (
            [mc_w(0),                               # dirt
             np.ones(len(skyshine_data)) * f_sky]   # skyshine — no GENIE weights
          + [mc_w(i) for i in range(1, len(mc_data))]  # remaining MC cats
          + [np.ones(len(off_beam_data)) * f_ext]   # off-beam
        )
    
    # 3. Aesthetics
    ### NEW: Added Skyshine to labels and colors and increased outlines to 7
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'darkmagenta', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALE
            if DEBUG:
                print(f"Applied {ALPHA_SCALE} scale to index {i} ({label})")
        ### NEW: Just a helpful printout to confirm the math worked
        if 'Skyshine' in label:
            if DEBUG:
                print(f"Scaled {len(skyshine_data)} raw skyshine events to exactly {expected_skyshine_yield:.1f} events")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill (softened with alpha so the hatch band / data points read clearly on top)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0, alpha=0.65)

    # B. MC stats uncertainty band, hatched, from the total stacked pred +/- pred_err
    x_step    = np.repeat(bins, 2)[1:-1]
    y_band_hi = np.repeat(pred_counts + pred_err, 2)
    y_band_lo = np.repeat(np.maximum(pred_counts - pred_err, 0), 2)
    ax.fill_between(x_step, y_band_lo, y_band_hi,
                     facecolor='none', edgecolor='dimgray',
                     linewidth=0, hatch='//', zorder=3,
                     label='MC stats')

    # C. Single bold outline of the total stacked prediction (replaces the
    #    per-component step outlines) -- ax.stairs requires matplotlib >= 3.4
    ax.stairs(pred_counts, bins, color='black', linewidth=1.8, zorder=4)

    # D. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data (stats only)', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) 
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.25, 1.02, 'Skyshine Sideband', weight = 'bold', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes, fontsize = 12)

    plt.tight_layout()
    if SAVE_PLOT:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()

    
print('done')

In [ ]:
common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'expected_skyshine_yield':SKYSHINE_FITTING_RESULTS,
    }

mc_weights_stack = [
    MC_weights_by_cat[5],
    MC_weights_by_cat[4],
    MC_weights_by_cat[2],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[0],
]


# energy
mc_stack = [MC_reco_E[5], MC_reco_E[4], MC_reco_E[2], MC_reco_E[1], MC_reco_E[3], MC_reco_E[0]]
plot_with_skyshine(
    bins=np.arange(5, 13, 1),
    mc_data=mc_stack,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=reco_E,
    off_beam_data=ext_reco_E,
    skyshine_data=skyshine_reco_E,
    axis_label='Reconstructed energy [MeV]',
    y_axis_label='Events / 1 MeV bin',
    x_limit=[5, 12],
    y_limit=[0,90],
    xaxis_minor=0.5,
    plot_name=plots_dir + 'skyshine_sideband_reco_energy.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)


mc_stack_hits = [MC_hits[5], MC_hits[4], MC_hits[2], MC_hits[1], MC_hits[3], MC_hits[0]]
plot_with_skyshine(
    bins=np.arange(10, 50, 4),
    mc_data=mc_stack_hits,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=hits,
    off_beam_data=ext_hits,
    skyshine_data=skyshine_hits,
    axis_label='PMT hits',
    y_axis_label='Events / 4 hit bin',
    x_limit=[10, 50],
    y_limit=[0,110],
    xaxis_minor=1,
    plot_name=plots_dir + 'skyshine_sideband_PMT_hits.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

mc_stack_r = [radius_mc[5], radius_mc[4], radius_mc[2], radius_mc[1], radius_mc[3], radius_mc[0]]
plot_with_skyshine(
    bins=np.arange(0, 0.8, 0.1),
    mc_data=mc_stack_r,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=radius_data,
    off_beam_data=radius_ext,
    skyshine_data=radius_skyshine,
    axis_label='Reconstructed distance to center [m]',
    y_axis_label='Events / 10 cm bin',
    x_limit=[0, 0.7],
    y_limit=[0,110],
    xaxis_minor=0.1,
    plot_name=plots_dir + 'skyshine_sideband_radius.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

mc_stack_y = [MC_vertex_y[5], MC_vertex_y[4], MC_vertex_y[2], MC_vertex_y[1], MC_vertex_y[3], MC_vertex_y[0]]
plot_with_skyshine(
    bins=np.arange(-1, 1.25, 0.25),
    mc_data=mc_stack_y,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=vertex_y,
    off_beam_data=ext_vertex_y,
    skyshine_data=skyshine_vertex_y,
    axis_label='Reconstructed vertex (Y) [m]',
    y_axis_label='Events / 50 cm bin',
    x_limit=[-1, 1],
    y_limit=[0,80],
    xaxis_minor=0.2,
    plot_name=plots_dir + 'skyshine_sideband_Y.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

vars_cv = [('x', MC_CV_x, CV_x, ext_CV_x, skyshine_CV_x),
           ('y', MC_CV_y, CV_y, ext_CV_y, skyshine_CV_y),
           ('z', MC_CV_z, CV_z, ext_CV_z, skyshine_CV_z)]
for name, mc_list, data_list, ext_list, sky_list in vars_cv:
    mc_stack = [mc_list[5], mc_list[4], mc_list[2], mc_list[1], mc_list[3], mc_list[0]]
    plot_with_skyshine(
        bins=np.arange(-1, 1.25, 0.25) if name != 'z' else np.arange(-1, 0.6, 0.2),
        mc_data=mc_stack,
        mc_weights_stack=mc_weights_stack,
        on_beam_data=data_list,
        off_beam_data=ext_list,
        skyshine_data=sky_list,
        axis_label=r'$C_V$ ' + f'({name})',
        y_axis_label='Events / bin',
        x_limit=[-1, 1] if name != 'z' else [-1, 0.4],
        y_limit=[0,80] if name != 'y' else [0, 100] ,
        xaxis_minor=0.1,
        plot_name=f'{plots_dir}skyshine_sideband_Cv_{name}.png',
        DEBUG=True,
        SAVE_PLOT=True,
        **common_args
    )
    

print('done')